In [308]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd() / "estonia-exam-gap"
RAW = ROOT / "data" / "raw"                # the downloaded Haridussilm / Statistikaamet files (never modified)
EXT = ROOT / "data" / "external"           # cached web look-ups (coordinates, road distances)
PROCESSED = ROOT / "data" / "processed"    # tables this notebook writes


In [309]:
!rm -rf ROOT
!git clone 'https://github.com/valpar/estonia-exam-gap.git'

fatal: destination path 'estonia-exam-gap' already exists and is not an empty directory.


# X1 : Qualified Teacher Ratio

Ideal Format (Columns)

school_name,
exam_year,
estonian_proficiency_ratio,
teaching_qualification_ratio,

In [310]:
df_raw_qualified_teacher = pd.read_excel(ROOT/RAW/'haridussilm_qualified_teachers_2021-2025_by_school.xlsx', header=2)
df_raw_qualified_teacher.head()

,Indikaator,Õppeasutus/KOV/Eesti,'Date'[Year],Väärtused
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,"95,5"
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100


In [311]:
# Examine the columns
df_raw_qualified_teacher.columns

Index(['Indikaator', 'Õppeasutus/KOV/Eesti', ''Date'[Year]', 'Väärtused'], dtype='object')

In [312]:
# Translation of estonian column names to english
df_raw_qualified_teacher = df_raw_qualified_teacher.rename(columns={
    "Õppeasutus/KOV/Eesti": "school",
    "'Date'[Year]": "year",
    "Indikaator": "indicator",
    "Väärtused": "percentage"
})
df_raw_qualified_teacher.head()

,indicator,school,year,percentage
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,"95,5"
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100


In [313]:
# Convert percentatge to decimal values
df_raw_qualified_teacher['percentage'] = df_raw_qualified_teacher['percentage'].astype(str).str.replace(',', '.').astype(float)

# Pivot the table to mathcing ideal format definted above
df_raw_qualified_teacher_wide = pd.pivot_table(
    df_raw_qualified_teacher,
    index=['school', 'year'],
    columns='indicator',
    values='percentage'
).reset_index()

# Rename the columns
df_raw_qualified_teacher_wide.columns = [
    'school_name',
    'exam_year',
    'estonian_proficiency_ratio',
    'teaching_qualification_ratio'
]

df_raw_qualified_teacher_wide.head()

,school_name,exam_year,estonian_proficiency_ratio,teaching_qualification_ratio
0,''Eduson'' kool,2021,80.0,100.0
1,''Eduson'' kool,2022,68.8,100.0
2,''Eduson'' kool,2023,60.0,100.0
3,''Eduson'' kool,2024,85.7,100.0
4,''Eduson'' kool,2025,100.0,100.0


# X2 : State of the School (State or Municipal)

Ideal Format (Columns)

school_name, ownership, academic_year

In [314]:
# Load schools ran by state
df_raw_school_state = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_state.xlsx', header=2)
df_raw_school_state.info()
df_raw_school_state.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   163 non-null    object
 1   Õppeasutus       163 non-null    object
 2   Õppurite arv ap  163 non-null    int64 
 3   Õppeaasta        163 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.2+ KB


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Haapsalu linn,Läänemaa Ühisgümnaasium,258,2020/21
1,Haapsalu linn,Läänemaa Ühisgümnaasium,272,2021/22
2,Haapsalu linn,Läänemaa Ühisgümnaasium,283,2022/23
3,Haapsalu linn,Läänemaa Ühisgümnaasium,260,2023/24
4,Haapsalu linn,Läänemaa Ühisgümnaasium,268,2024/25


In [315]:
# Load schools ran by municipal
df_raw_school_municipal = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_municipal.xlsx', header=2)
df_raw_school_municipal.info()
df_raw_school_municipal.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   631 non-null    object
 1   Õppeasutus       631 non-null    object
 2   Õppurite arv ap  631 non-null    int64 
 3   Õppeaasta        631 non-null    object
dtypes: int64(1), object(3)
memory usage: 19.8+ KB


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,Iisaku Põhikool,7,2024/25


In [316]:
# Add ownership label for two different type of school
df_raw_school_state.insert(1, 'ownership', value='state')
df_raw_school_municipal.insert(1, 'ownership', value='municipal')

# Merge school names (Õppeasutus) from both table and give state/municipal labels
df_raw_school_combined = pd.concat([df_raw_school_municipal, df_raw_school_state], ignore_index=True)
df_raw_school_combined.info()
df_raw_school_combined.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   794 non-null    object
 1   ownership        794 non-null    object
 2   Õppeasutus       794 non-null    object
 3   Õppurite arv ap  794 non-null    int64 
 4   Õppeaasta        794 non-null    object
dtypes: int64(1), object(4)
memory usage: 31.1+ KB


,KOV/Õppeasutus,ownership,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,municipal,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,municipal,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,municipal,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,municipal,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,municipal,Iisaku Põhikool,7,2024/25


In [317]:
# Rename table column propery
df_raw_school_combined = df_raw_school_combined[['Õppeasutus', 'ownership','Õppeaasta']]
df_raw_school_combined.columns = [
    'school_name',
    'ownership',
    'academic_year'
]
df_raw_school_combined.head()

,school_name,ownership,academic_year
0,Iisaku Põhikool,municipal,2020/21
1,Iisaku Põhikool,municipal,2021/22
2,Iisaku Põhikool,municipal,2022/23
3,Iisaku Põhikool,municipal,2023/24
4,Iisaku Põhikool,municipal,2024/25




# X3 : Distance to Hubs

Ideal Format (Columns)

school_name, distance_to_tallinn, distance_to_tartu



# X4 : Teaching Language / Exam Langauge

Ideal Format (Columns)

school_name, academic_year, lang_insruction	take_estonian_exam, take_exam_in_ukranian	take_exam_in_russian

In [318]:
# Loading the school language of instruction data
df_raw_lang_instruction = pd.read_excel(ROOT/RAW/'riigieksam_math_2021-2025_by_school_language.xlsx', header=2)
df_raw_lang_instruction.info()
df_raw_lang_instruction.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1430 entries, 0 to 1429
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Õppeaasta                  1430 non-null   object 
 1   Eksam                      1430 non-null   object 
 2   Õppeasutus                 1430 non-null   object 
 3   Õppekeel                   1430 non-null   object 
 4   Max võimalik punktide arv  1430 non-null   int64  
 5   Sooritajate arv            1430 non-null   int64  
 6   Keskmine                   1241 non-null   float64
 7   Keskmine (%)               1241 non-null   float64
 8   St. Hälve                  1241 non-null   float64
 9   Mediaan                    1241 non-null   float64
 10  AUP                        1241 non-null   float64
 11  ÜUP                        1241 non-null   float64
 12  Min                        1241 non-null   float64
 13  Min punktide saajaid       25 non-null     float

,Õppeaasta,Eksam,Õppeasutus,Õppekeel,Max võimalik punktide arv,Sooritajate arv,Keskmine,Keskmine (%),St. Hälve,Mediaan,AUP,ÜUP,Min,Min punktide saajaid,Max,Max punktide saajaid,B2 (%)
0,2024/25,Matemaatika (riigieksam) (kitsas),Abja Gümnaasium,eesti keel,100,13,35.153846,0.351538,19.398652,28.0,24.608619,45.699073,14.0,NaN,77.0,NaN,NaN
1,2024/25,Matemaatika (riigieksam) (kitsas),Antsla Gümnaasium,eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasium,eesti keel,100,15,36.400000,0.364000,16.565023,37.0,28.016942,44.783058,6.0,NaN,72.0,NaN,NaN
3,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasiumi Otepää filiaal,eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024/25,Matemaatika (riigieksam) (kitsas),August Kitzbergi nimeline Gümnaasium,eesti keel,100,7,34.142857,0.341429,18.836388,25.0,20.188665,48.097049,19.0,NaN,69.0,NaN,NaN


In [319]:
# Extract Math(lai) only
df_raw_lang_instruction = df_raw_lang_instruction[df_raw_lang_instruction['Eksam'].str.contains('lai')]

# Extract only school names and their languages of instruction
df_raw_lang_instruction = df_raw_lang_instruction[['Õppeasutus', 'Õppeaasta', 'Õppekeel']]
df_raw_lang_instruction.info()
df_raw_lang_instruction.head()

<class 'pandas.core.frame.DataFrame'>
Index: 715 entries, 141 to 1429
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Õppeasutus  715 non-null    object
 1   Õppeaasta   715 non-null    object
 2   Õppekeel    715 non-null    object
dtypes: object(3)
memory usage: 22.3+ KB


,Õppeasutus,Õppeaasta,Õppekeel
141,Abja Gümnaasium,2024/25,eesti keel
142,Antsla Gümnaasium,2024/25,eesti keel
143,Audentese Spordigümnaasium,2024/25,eesti keel
144,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel
145,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel


In [320]:
# Translation of estonian column names to english
df_raw_lang_instruction = df_raw_lang_instruction.rename(columns={
    "Õppeaasta": "academic_year",
    "Õppeasutus": "school_name",
    "Õppekeel": "lang_insruction"
})
df_raw_lang_instruction.head()

,school_name,academic_year,lang_insruction
141,Abja Gümnaasium,2024/25,eesti keel
142,Antsla Gümnaasium,2024/25,eesti keel
143,Audentese Spordigümnaasium,2024/25,eesti keel
144,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel
145,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel


In [321]:
# Loading the language of exam data
df_raw_lang_exam = pd.read_excel(ROOT/RAW/'riigieksam_math_2021-2025_by_school_examlanguage.xlsx', header=2)
df_raw_lang_exam.info()
df_raw_lang_exam.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1513 entries, 0 to 1512
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Õppeaasta                  1513 non-null   object 
 1   Eksam                      1513 non-null   object 
 2   Õppeasutus                 1513 non-null   object 
 3   Soorituskeel               1513 non-null   object 
 4   Max võimalik punktide arv  1513 non-null   int64  
 5   Sooritajate arv            1513 non-null   int64  
 6   Keskmine                   1258 non-null   float64
 7   Keskmine (%)               1258 non-null   float64
 8   St. Hälve                  1258 non-null   float64
 9   Mediaan                    1258 non-null   float64
 10  AUP                        1258 non-null   float64
 11  ÜUP                        1258 non-null   float64
 12  Min                        1258 non-null   float64
 13  Min punktide saajaid       24 non-null     float

,Õppeaasta,Eksam,Õppeasutus,Soorituskeel,Max võimalik punktide arv,Sooritajate arv,Keskmine,Keskmine (%),St. Hälve,Mediaan,AUP,ÜUP,Min,Min punktide saajaid,Max,Max punktide saajaid,B2 (%)
0,2024/25,Matemaatika (riigieksam) (kitsas),Abja Gümnaasium,Eesti keel,100,13,35.153846,0.351538,19.398652,28.0,24.608619,45.699073,14.0,NaN,77.0,NaN,NaN
1,2024/25,Matemaatika (riigieksam) (kitsas),Antsla Gümnaasium,Eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasium,Eesti keel,100,15,36.400000,0.364000,16.565023,37.0,28.016942,44.783058,6.0,NaN,72.0,NaN,NaN
3,2024/25,Matemaatika (riigieksam) (kitsas),Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024/25,Matemaatika (riigieksam) (kitsas),August Kitzbergi nimeline Gümnaasium,Eesti keel,100,7,34.142857,0.341429,18.836388,25.0,20.188665,48.097049,19.0,NaN,69.0,NaN,NaN


In [322]:
# Extract Math(lai) only
df_raw_lang_exam = df_raw_lang_exam [df_raw_lang_exam['Eksam'].str.contains('lai')]

# Extract only school names and their languages of instruction
df_raw_lang_exam = df_raw_lang_exam[['Õppeaasta', 'Õppeasutus', 'Soorituskeel']]

df_raw_lang_exam.info()
df_raw_lang_exam.head()

<class 'pandas.core.frame.DataFrame'>
Index: 758 entries, 155 to 1512
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Õppeaasta     758 non-null    object
 1   Õppeasutus    758 non-null    object
 2   Soorituskeel  758 non-null    object
dtypes: object(3)
memory usage: 23.7+ KB


,Õppeaasta,Õppeasutus,Soorituskeel
155,2024/25,Abja Gümnaasium,Eesti keel
156,2024/25,Antsla Gümnaasium,Eesti keel
157,2024/25,Audentese Spordigümnaasium,Eesti keel
158,2024/25,Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel
159,2024/25,August Kitzbergi nimeline Gümnaasium,Eesti keel


In [323]:
# Translation of estonian column names to english
df_raw_lang_exam = df_raw_lang_exam.rename(columns={
    "Õppeaasta": "academic_year",
    "Õppeasutus": "school_name",
    "Soorituskeel": "exam_language"
})
df_raw_lang_exam.head()

,academic_year,school_name,exam_language
155,2024/25,Abja Gümnaasium,Eesti keel
156,2024/25,Antsla Gümnaasium,Eesti keel
157,2024/25,Audentese Spordigümnaasium,Eesti keel
158,2024/25,Audentese Spordigümnaasiumi Otepää filiaal,Eesti keel
159,2024/25,August Kitzbergi nimeline Gümnaasium,Eesti keel


In [324]:
# Adding extra column as language indicator
df_raw_lang_exam.insert(3, 'flag', value=1)
df_raw_lang_exam.head()

# Pivot the table to mathcing ideal format definted above
df_raw_lang_exam_wide = pd.pivot_table(
    df_raw_lang_exam,
    index=['school_name', 'academic_year'],
    columns='exam_language',
    values='flag'
).reset_index()

df_raw_lang_exam_wide.info()
df_raw_lang_exam_wide.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   school_name    708 non-null    object 
 1   academic_year  708 non-null    object 
 2   Eesti keel     580 non-null    float64
 3   Ukraina keel   2 non-null      float64
 4   Vene keel      176 non-null    float64
dtypes: float64(3), object(2)
memory usage: 27.8+ KB


exam_language,school_name,academic_year,Eesti keel,Ukraina keel,Vene keel
0,''Eduson'' kool,2022/23,NaN,NaN,1.0
1,Abja Gümnaasium,2020/21,1.0,NaN,NaN
2,Abja Gümnaasium,2021/22,1.0,NaN,NaN
3,Abja Gümnaasium,2022/23,1.0,NaN,NaN
4,Abja Gümnaasium,2023/24,1.0,NaN,NaN


In [325]:
# Fill missing flag with 0
df_raw_lang_exam_wide['Eesti keel'] = df_raw_lang_exam_wide['Eesti keel'].fillna(0)
df_raw_lang_exam_wide['Ukraina keel'] = df_raw_lang_exam_wide['Ukraina keel'].fillna(0)
df_raw_lang_exam_wide['Vene keel'] = df_raw_lang_exam_wide['Vene keel'].fillna(0)

df_raw_lang_exam_wide.columns = [
    'school_name',
    'academic_year',
    'take_estonian_exam',
    'take_exam_in_ukranian',
    'take_exam_in_russian'
]
df_raw_lang_exam_wide.info()
df_raw_lang_exam_wide.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   school_name            708 non-null    object 
 1   academic_year          708 non-null    object 
 2   take_estonian_exam     708 non-null    float64
 3   take_exam_in_ukranian  708 non-null    float64
 4   take_exam_in_russian   708 non-null    float64
dtypes: float64(3), object(2)
memory usage: 27.8+ KB


,school_name,academic_year,take_estonian_exam,take_exam_in_ukranian,take_exam_in_russian
0,''Eduson'' kool,2022/23,0.0,0.0,1.0
1,Abja Gümnaasium,2020/21,1.0,0.0,0.0
2,Abja Gümnaasium,2021/22,1.0,0.0,0.0
3,Abja Gümnaasium,2022/23,1.0,0.0,0.0
4,Abja Gümnaasium,2023/24,1.0,0.0,0.0


In [326]:
# Combine two table regarding used languages
df_used_language_combined = df_raw_lang_instruction.merge(df_raw_lang_exam_wide, how='inner', on=['school_name', 'academic_year'])
df_used_language_combined.head()

,school_name,academic_year,lang_insruction,take_estonian_exam,take_exam_in_ukranian,take_exam_in_russian
0,Abja Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0
1,Antsla Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0
2,Audentese Spordigümnaasium,2024/25,eesti keel,1.0,0.0,0.0
3,Audentese Spordigümnaasiumi Otepää filiaal,2024/25,eesti keel,1.0,0.0,0.0
4,August Kitzbergi nimeline Gümnaasium,2024/25,eesti keel,1.0,0.0,0.0
